# Hate Speech Detection (Sinhala-English, Facebook + YouTube) - Full Colab Run

Run every cell **top to bottom**. First: Runtime -> Change runtime type -> **T4 GPU**.

Both conditions (no emoji / with emoji) use the **same data rows, same folds, same settings**. Only `--condition` and `--data` differ.
Use ONE results folder (`OUTDIR`, set in cell 2) for the whole run. Never reuse an old folder.

In [ ]:
# Cell 1 - GPU check
import torch
print('CUDA available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'No GPU. Runtime -> Change runtime type -> T4 GPU, restart, re-run.'

In [ ]:
# Cell 2 - Mount Drive and set the ONE results folder used by every later cell
from google.colab import drive
drive.mount('/content/drive')
import os
OUTDIR = '/content/drive/MyDrive/hate_speech_final_run'
os.makedirs(OUTDIR, exist_ok=True)
print('Results folder:', OUTDIR)
print('Existing contents:', os.listdir(OUTDIR), '(should be empty for a fresh run)')

In [ ]:
# Cell 3 - Upload the project zip
import os, zipfile
if os.path.exists('/content/project'):
    print('project/ already exists; to refresh run: !rm -rf /content/project')
else:
    from google.colab import files
    up = files.upload()
    name = list(up.keys())[0]
    with zipfile.ZipFile(name) as z: z.extractall('/content')
    print('Extracted', name)
print(os.listdir('/content'))

In [ ]:
# Cell 4 - Verify required files and that both variants have identical rows
import os, pandas as pd, json
P = '/content/project'
D = f'{P}/1_Data_Preparation'
T = f'{P}/2_Model_Training'
E = f'{P}/3_Evaluation_and_Stats'
need = [f'{D}/data_processed/data_no_emoji.csv', f'{D}/data_processed/data_with_emoji.csv',
        f'{D}/data_processed/folds.json', f'{T}/train.py']
miss = [p for p in need if not os.path.exists(p)]
assert not miss, f'Missing: {miss}'
a = pd.read_csv(f'{D}/data_processed/data_with_emoji.csv'); b = pd.read_csv(f'{D}/data_processed/data_no_emoji.csv')
assert len(a) == len(b) and (a.comment_id.values == b.comment_id.values).all(), 'Variants are misaligned!'
print('OK -', len(a), 'rows in both variants'); print(a.label.value_counts()); print(pd.crosstab(a.source, a.label))

In [ ]:
# Cell 5 - Install dependencies
!pip install -q -r /content/project/requirements.txt

In [ ]:
# Cell 6 - BASELINE (no emoji), all 5 folds. Saves to Drive after every fold.
%cd /content/project/2_Model_Training
!python train.py --condition no_emoji \
    --data ../1_Data_Preparation/data_processed/data_no_emoji.csv \
    --folds ../1_Data_Preparation/data_processed/folds.json \
    --outdir {OUTDIR} \
    --epochs 10 --patience 3 --seed 42

In [ ]:
# Cell 7 - PROPOSED (with emoji). ONLY --condition and --data differ from cell 6.
%cd /content/project/2_Model_Training
!python train.py --condition with_emoji \
    --data ../1_Data_Preparation/data_processed/data_with_emoji.csv \
    --folds ../1_Data_Preparation/data_processed/folds.json \
    --outdir {OUTDIR} \
    --epochs 10 --patience 3 --seed 42

In [ ]:
# Cell 8 - Aggregate + Wilcoxon (checks both conditions finished all 5 folds)
import os, shutil, glob
LOCAL = '/content/project/3_Evaluation_and_Stats/outputs_final'
for cond in ['no_emoji', 'with_emoji']:
    n = len(glob.glob(f'{OUTDIR}/{cond}/fold_*/metrics.json'))
    done = os.path.exists(f'{OUTDIR}/{cond}/all_folds_metrics.json')
    print(f'{cond}: {n}/5 folds | all_folds_metrics.json: {done}')
    if n != 5 or not done:
        raise SystemExit(f'{cond} not finished - re-run its training cell.')
if os.path.exists(LOCAL): shutil.rmtree(LOCAL)
shutil.copytree(OUTDIR, LOCAL, ignore=shutil.ignore_patterns('model', '*.safetensors', '*.bin'))
%cd /content/project/3_Evaluation_and_Stats
!python evaluate.py --outdir {LOCAL} --savedir {LOCAL}/summary
print()
!python stats_test.py --outdir {LOCAL} --savedir {LOCAL}/summary

In [ ]:
# Cell 9 - Reference baseline: TF-IDF + Logistic Regression on the SAME folds (sanity check / extra table row)
import pandas as pd, json, numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import f1_score
P = '/content/project/1_Data_Preparation/data_processed'
folds = json.load(open(f'{P}/folds.json'))['folds']
for name in ['no_emoji', 'with_emoji']:
    df = pd.read_csv(f'{P}/data_{name}.csv'); s = []
    for fo in folds:
        tr, te = df.iloc[fo['train_rows']], df.iloc[fo['test_rows']]
        m = make_pipeline(TfidfVectorizer(analyzer='char_wb', ngram_range=(2,5), min_df=2),
                          LogisticRegression(max_iter=2000, class_weight='balanced'))
        m.fit(tr.clean_text.fillna(''), tr.label)
        s.append(f1_score(te.label, m.predict(te.clean_text.fillna('')), average='macro'))
    print(f'TF-IDF+LR {name}: macro-F1 {np.mean(s):.3f} +/- {np.std(s):.3f}  per fold {np.round(s,3)}')

In [ ]:
# Cell 10 - Zip results (without model weights) and download
%cd /content/project/3_Evaluation_and_Stats
!zip -r outputs_final.zip outputs_final > /dev/null
from google.colab import files
files.download('outputs_final.zip')

### After it finishes
Check `summary_metrics_full.json` (per-class F1, confusion matrices) for any fold where the model predicts a single class (collapse: macro-F1 ~0.19). Apply the same retraining rule to BOTH conditions if so, and report it.